## Pipeline Layers

1. **Raw Table**: Ingest Wikipedia clickstream JSON data
2. **Cleansed Table**: Apply quality expectations to filter bad records
3. **Derived Table**: Aggregate for analysis

## Key Terms

| Term | What it means in your code |
|---|---|
| `STREAM` | Read data incrementally (new files only) |
| `LIVE.` | Reference another dataset in this pipeline |
| `EXPECT` | Data quality rule: drop, warn, or fail |
| `MATERIALIZED VIEW` | Pre-computed results, refreshed by the pipeline |

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE clickstream_raw
COMMENT "Raw Wikipedia clickstream data ingested from JSON files."
AS
SELECT * FROM STREAM read_files(
  "/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed-json",
  format => 'json'
);

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE clickstream_clean(
  CONSTRAINT valid_current_page EXPECT (curr_title IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_click_count EXPECT (n > 0) ON VIOLATION DROP ROW
)
COMMENT "Cleaned Wikipedia clickstream data with quality checks applied."
AS
SELECT
  curr_title AS current_page_title,
  prev_title AS previous_page_title,
  CAST(n AS INT) AS click_count,
  type AS link_type
FROM STREAM(live.clickstream_raw);

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW top_clickstream_pages
COMMENT "Aggregated click counts by destination page."
AS
SELECT
  current_page_title,
  SUM(click_count) AS total_clicks,
  COUNT(*) AS link_types_count
FROM live.clickstream_clean
GROUP BY current_page_title
ORDER BY total_clicks DESC;
-- We use MATERIALIZED VIEW because this is an aggregate that benefits from being pre-computed. The live.clickstream_clean reference creates an automatic dependency — DLT knows to run the cleansed table first

## What We Practiced

- Built a 3-layer pipeline: **raw → clean → aggregated**
- Used `EXPECT` constraints to drop bad rows (`curr_title IS NOT NULL`, `n > 0`)
- Learned that `EXPECT` runs on source columns, so it needs the raw column names

Where you're running	
How to reference a table	

Example
Notebook=	Full path	    workspace.default.clickstream_raw
Pipeline=	live. prefix	live.clickstream_raw

In [0]:
# import dlt
# from pyspark.sql.functions import col

# # Cell 1: Raw ingestion
# @dlt.table(
#     name="clickstream_raw_py",
#     comment="Raw Wikipedia clickstream data."
# )
# def clickstream_raw():
#     return (
#         spark.readStream.format("cloudFiles")
#             .option("cloudFiles.format", "json")
#             .load("/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed-json")
#             .select(
#                 col("curr_title").alias("current_page_title"),
#                 col("prev_title").alias("previous_page_title"),
#                 col("n").cast("int").alias("click_count"),
#                 col("type").alias("link_type")
#             )
#     )

# # Cell 2: Cleaned with expectations
# @dlt.table(
#     name="clickstream_clean_py",
#     comment="Cleaned Wikipedia clickstream data."
# )
# @dlt.expect_or_drop("valid_current_page", "current_page_title IS NOT NULL")
# @dlt.expect_or_drop("valid_click_count", "click_count > 0")
# def clickstream_clean():
#     return dlt.read_stream("clickstream_raw_py")

# # Cell 3: Aggregated view
# @dlt.table(
#     name="top_clickstream_pages_py",
#     comment="Aggregated click counts."
# )
# def top_clickstream_pages():
#     return (
#         dlt.read("clickstream_clean_py")
#             .groupBy("current_page_title")
#             .agg(
#                 {"click_count": "sum"}
#             )
#             .withColumnRenamed("sum(click_count)", "total_clicks")
#     )

## Parameterization

Define key-value pairs in **Pipeline Settings → Configuration**.

| Language | How to read a parameter |
|---|---|
| SQL | `${param_name}` |
| Python | `spark.conf.get("param_name")` |

**Use case:** same code, different environments (dev/test/prod).